## **Libraries**

In [ ]:
import os
import cv2
import pandas as pd
from ultralytics import YOLO
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
import plotly.io as pio
pio.renderers.default = "svg"

## **Configurations**

In [ ]:
# Paths
base_dir = os.path.dirname(os.path.abspath(__file__))
model_path = os.path.join(base_dir, "output", "weights", "best.pt")
input_video = os.path.join(base_dir, "input", "cali_fire.mp4") 
output_dir = os.path.join(base_dir, "output", "tracking")  
os.makedirs(os.path.dirname(output_dir), exist_ok=True)
output_vid = os.path.join(output_dir, "track_output.mp4")

# Variables
img_size = 640
device = 0
tracker = "bytetrack.yaml"
track_conf = 0.10

## **Open The Video**

In [ ]:
cap = cv2.VideoCapture(str(input_video))

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {input_video}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0 or width <= 0 or height <= 0 or total_frames <= 0:
    cap.release()
    raise RuntimeError("Could not read valid video metadata")

print(f"FPS: {fps}")
print(f"Resolution: {width} x {height}")
print(f"Frames: {total_frames}")

FPS: 25.0
Resolution: 640 x 360
Frames: 1233


## **Inspecting One Frame**

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

success, frame = cap.read()

if not success:
    raise RuntimeError("Could not read the first frame")

print(f"Shape: {frame.shape}")

cv2.imshow("First Frame", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

Shape: (360, 640, 3)


## **Run Prediction By YOLO**

In [ ]:
model = YOLO(str(model_path))

res = model.predict(
    frame,
    imgsz=img_size,
    conf=track_conf,
    device=device,
    verbose=False
)[0]

In [ ]:
fig1 = make_subplots(rows=1, cols=1)

draw_res = cv2.cvtColor(
    res.plot(),
    cv2.COLOR_BGR2RGB
)

fig1.add_trace(go.Image(z=draw_res), row=1, col=1)

fig1.update_layout(
    template= "plotly_dark",
    title= "Inspecting Detection Of The First Frame",
    title_x= 0.5
)

fig1.show()

## **Adding ByteTrack**

In [ ]:
tracking_model = YOLO(str(model_path))

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
success, tracking_frame = cap.read()

if not success:
    raise RuntimeError("Could not read the first frame for tracking")

tracking_res = tracking_model.track(
    tracking_frame,
    persist=True,
    tracker=tracker,
    imgsz=img_size,
    conf=track_conf,
    device=device,
    verbose=False
)[0]

In [ ]:
fig2 = make_subplots(rows=1, cols=1)

draw_res2 = cv2.cvtColor(
    tracking_res.plot(),
    cv2.COLOR_BGR2RGB
)

fig2.add_trace(go.Image(z=draw_res2), row=1, col=1)

fig2.update_layout(
    template="plotly_dark",
    title="Inspecting Tracking Of The First Frame",
    title_x=0.5
)

fig2.show()

## **Inspecting 2 Frames**

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_tracking_model = YOLO(str(model_path))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Frame 1", "Frame 2"],
    horizontal_spacing=0.08,
    vertical_spacing=0.08
)

for col in range(1, 3):
    success, frame = cap.read()

    if not success:
        raise RuntimeError(f"Could not read frame {col}")

    track_res = frame_tracking_model.track(
        frame,
        persist=True,
        tracker=tracker,
        imgsz=img_size,
        conf=track_conf,
        device=device,
        verbose=False
    )[0]

    draw_res = cv2.cvtColor(
        track_res.plot(),
        cv2.COLOR_BGR2RGB
    )

    fig.add_trace(
        go.Image(z=draw_res),
        row=1,
        col=col
    )

fig.update_layout(
    template="plotly_dark",
    title="Inspecting Tracking Of 2 Frames",
    title_x=0.5,
    height=600,
    width=1100,
    margin=dict(t=80)
)

fig.show()

## **Inspecting a Small Sequence**

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
sequence_model = YOLO(str(model_path))
detection_history = []

for frame_num in range(1, min(30, total_frames) + 1):
    success, frame = cap.read()

    if not success:
        raise RuntimeError(f"Could not read frame {frame_num}")

    result = sequence_model.track(
        frame,
        persist=True,
        device=device,
        tracker=tracker,
        conf=track_conf,
        imgsz=img_size,
        verbose=False
    )[0]

    if result.boxes.id is None:
        continue

    ids = result.boxes.id.cpu().numpy().astype(int)
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()

    for track_id, class_id, confidence in zip(ids, classes, confs):
        detection_history.append({
            "Frame": frame_num,
            "Track_ID": int(track_id),
            "Class_ID": int(class_id),
            "Confidence": float(confidence)
        })

In [ ]:
hist_df = pd.DataFrame(detection_history)
hist_df

,Frame,Track_ID,Class_ID,Confidence
0,1,1,1,0.649813
1,1,2,0,0.648631
2,1,3,1,0.593476
3,1,4,1,0.559779
4,1,5,1,0.544915
...,...,...,...,...
243,30,4,1,0.415954
244,30,5,1,0.154540
245,30,8,1,0.250661
246,30,11,0,0.222776


In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    str(output_vid),
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not create output video:\n{output_vid}")

In [ ]:
while True:
    success, frame = cap.read()

    if not success:
        break

    frame_num += 1
    processed_frames += 1

    results = model.track(
        frame,
        persist=True,
        device=device,
        tracker=tracker,
        conf=track_conf,
        imgsz=img_size,
        verbose=False
    )[0]

    if results.boxes is not None and results.boxes.id is not None:
        boxes = results.boxes

        ids = boxes.id.cpu().numpy().astype(int)
        class_ids = boxes.cls.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()

        for track_id, class_id, bbox, conf in zip(ids, class_ids, xyxy, confidence):
            x1, y1, x2, y2 = bbox.astype(int)
            class_name = results.names[class_id]
        
            color = (0, 0, 255) if class_name == "fire" else (255, 0, 0)

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            label = f"{class_name} {confidence:.2f}"

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
                cv2.LINE_AA
            )

    writer.write(frame)
    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("Stopped by the user")
        break

cap.release()
writer.release()
cv2.destroyAllWindows()